In [27]:
# Download do dataset


import kagglehub


print('Baixando dataset...')
path = kagglehub.dataset_download("svanoo/myanimelist-dataset")

print("Caminho até o dataset:", path)


Baixando dataset...
Caminho até o dataset: /root/.cache/kagglehub/datasets/svanoo/myanimelist-dataset/versions/2


In [28]:
# Import das bibliotecas necessárias


import string
import nltk
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
# Input de dados


path = path + '/anime.csv'

df_anime = pd.read_csv(path, sep='\t')

In [30]:
# Filtrando colunas de interesse


df_anime = df_anime.iloc[:, [2, 3, 12, 13]]

In [31]:
# Inicializando dicionário e lista de sinopses para tratamento dos dados


anime_base = {}


for title in df_anime['title']:
    anime_base[title] = None


synopsis_list = [synopsis for synopsis in df_anime['synopsis']]


In [32]:
# Criando listas de tokens


tokenized_lists = []


for index, synopsis in enumerate(synopsis_list):
  if isinstance(synopsis, float):
      synopsis_list[index] = ''


for index, synopsis in enumerate(synopsis_list):
        tokenized_lists.append(nltk.tokenize.word_tokenize(synopsis))


In [33]:
# Remoção de Stopwords


tokenized_list_clean_1 = []


for list_of_tokens in tokenized_lists:
    tokenized_sublist = []
    for token in list_of_tokens:
        if not token.lower() in nltk.corpus.stopwords.words('english') and not token in string.punctuation:
            tokenized_sublist.append(token)
    tokenized_list_clean_1.append(tokenized_sublist)


In [35]:
# Remoção de mais alguns ruídos (listas vazias, reticiências, aspas simples)


tokenized_list_clean_2 = []

remanescent_data_noise = ['...', "''", '``', '-', '’', "'s", "n't", "'ll"]


for list_of_tokens in tokenized_list_clean_1:
    tokenized_sublist = []
    for token in list_of_tokens:
        if not token in remanescent_data_noise:
            tokenized_sublist.append(token)
    tokenized_list_clean_2.append(tokenized_sublist)


print(tokenized_list_clean_2)


Output hidden; open in https://colab.research.google.com to view.

In [36]:
# Tratamento de dados da coluna GÊNERO


genre_list = []


for index, genre in enumerate(df_anime['genres']):
    if isinstance(genre, str):
        genre_list.append(genre)
        if '|' in genre_list[index]:
            genre_list[index] = genre_list[index].split('|')
    else:
        genre_list.append('')


print(genre_list)


['Supernatural', ['Action', 'Adventure'], 'Comedy', ['Comedy', 'Slice of Life'], ['Action', 'Adventure', 'Kids'], 'Slice of Life', ['Comedy', 'Kids'], ['Comedy', 'Kids'], ['Comedy', 'Kids'], ['Sci-Fi', 'Sports', 'Super Power', 'Kids'], ['Fantasy', 'Kids'], ['Adventure', 'Space'], ['Action', 'Adventure', 'Comedy', 'Fantasy', 'Romance'], ['Action', 'Mecha'], ['Gourmet', 'Slice of Life', 'Kids'], 'Historical', ['Comedy', 'Slice of Life', 'Cars', 'Kids'], ['Action', 'Fantasy', 'Demons', 'School'], ['Action', 'School'], ['Comedy', 'Sci-Fi'], ['Comedy', 'Fantasy', 'Slice of Life'], ['Comedy', 'Romance'], ['Music', 'Kids'], ['Action', 'Sci-Fi', 'Mecha'], 'Comedy', ['Adventure', 'Cars', 'Kids'], ['Comedy', 'Music'], 'Comedy', ['Comedy', 'Supernatural'], ['Action', 'Adventure', 'Comedy', 'Fantasy', 'Romance', 'Harem'], 'Kids', ['Action', 'Adventure', 'Comedy', 'Romance', 'Sci-Fi'], ['Comedy', 'Drama', 'Slice of Life', 'School', 'Shounen'], 'Comedy', ['Slice of Life', 'Kids'], ['Comedy', 'Histor

In [37]:
# Tratamento de dados da coluna ESTÚDIO


studio_list = []


for index, genre in enumerate(df_anime['studios']):
    if isinstance(genre, str):
        studio_list.append(genre)
        if '|' in studio_list[index]:
            studio_list[index] = studio_list[index].split('|')
    else:
        studio_list.append('')


print(studio_list)


['J.C.Staff', '', ['Space Neko Company', 'Jinnan Studio'], ['Toei Animation', 'Daewon Media'], '', '', '', '', 'DLE', '', '', ['Studio W.Baba', 'P.I.C.S.'], '', 'Production Reed', 'Digital Media Lab', '', 'CloverWorks', 'SILVER LINK.', 'Bibury Animation Studios', 'Fanworks', 'Doga Kobo', '', '', 'Toei Animation', 'FIREBUG', '', '', '', '', '', '', 'Magic Bus', '', 'Kachidoki Studio', 'Kachidoki Studio', '', 'Toei Animation', '', '', ['Ajia-Do', 'TMS Entertainment'], '', '', '', '', '', '', 'Studio 3Hz', 'TMS Entertainment', 'P.A. Works', 'LIDENFILMS', ['Toei Animation', 'Gallop'], ['Nippon Animation', 'OLM'], '', 'DLE', 'Dazzling Star', '', '', '', 'Doga Kobo', '', '', '', '', '', 'Kate Arrow', '', 'Alpha Animation', '', ["Brain's Base", 'Shogakukan Music & Digital Entertainment'], 'Shanghai Animation Film Studio', '', '', ['Eiken', 'Studio Live'], '', '', '', 'Taomee', '', 'Knack Productions', 'MAPPA', 'Sunrise', '', 'Milky Cartoon', '33 Collective', '', '', '', '', 'Beijing Rocen Dig

In [39]:
# Agrupando todas as informações em uma única lista


for index, synopsis in enumerate(tokenized_list_clean_2):
    if isinstance(genre_list[index], list):
        for genre in genre_list[index]:
            tokenized_list_clean_2[index].append(genre)
    else:
        tokenized_list_clean_2[index].append(genre_list[index])
    if isinstance(studio_list[index], list):
        for studio in studio_list[index]:
            tokenized_list_clean_2[index].append(studio)
    elif studio_list[index] == '':
        continue
    else:
        tokenized_list_clean_2[index].append(studio_list[index])

print(tokenized_list_clean_2[:5])


[['Shuramaru', 'hated', 'feared', 'villagers', 'unusual', 'powers', 'thinks', 'human', 'Source', 'AniDB', 'Supernatural', 'J.C.Staff'], ['Sometime', 'future', 'world', 'completely', 'dried', 'became', 'desert', 'little', 'rivers', 'lakes', 'left', 'villians', 'dangerous', 'animals', 'lived', 'Water', 'become', 'valueable', 'thing', 'world', 'Whoever', 'control', 'water', 'rule', 'world', 'Source', 'ANN', 'Action', 'Adventure'], ['Set', '2014', 'anime', 'follows', 'adventures', '23', 'years', 'old', 'Mafuneko', 'newly', 'minted', 'assistant', 'director', 'joins', 'TV', 'production', 'department', 'Tokyo', 'Hajikko', 'Television', 'discover', 'glamorous', 'glitzy', 'life', 'working', 'behind-the-scenes', 'making', 'TV', 'shows', 'involves', 'strange', 'inexplicable', 'tasks', 'gathering', '300', 'acorns', 'making', 'mosaic', 'images', 'reflected', 'camera', 'lens', 'Despite', 'surrounded', 'chaos', 'set-backs', 'weirdos', 'Mafuneko', 'struggles', 'become', 'fully-fledged', 'TV', 'produce

In [ ]:
# Inserindo as listas de tokens no dicionário de animes


index = 0

for anime in anime_base:
    anime_base[anime] = tokenized_list_clean_2[index]
    index =+ 1


13379
